In [10]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import get_project_root
print(get_project_root())
from pathlib import Path
from src.config import get_project_root


2026-09-20 22:56:21,660 - INFO - Project root resolved to: D:\MlOps_tasks\task_3


D:\MlOps_tasks\task_3


In [11]:

import pandas as pd, yaml
PROJ=get_project_root()
p=yaml.safe_load(open(PROJ/"config"/'params.yaml'))
train = pd.read_csv(p['paths']['ml_train_features'])
val = pd.read_csv(p['paths']['ml_val_features'])
test = pd.read_csv(p['paths']['ml_test_features'])

############################ convert one-hot cols from float to int since all of them are 0/1
onehot_cols = [c for c in train.columns if c.startswith(('sel_state_','cat_','pay_','cust_state_'))]
for df in [train, val, test]:
    df[onehot_cols] = df[onehot_cols].astype(int)

print(train.shape, val.shape, test.shape)
print(train.dtypes.head(60))

2026-09-20 22:56:23,434 - INFO - Project root resolved to: D:\MlOps_tasks\task_3


(67533, 118) (9647, 118) (19296, 118)
num_sellers                        float64
num_items                          float64
total_price                        float64
total_freight_value                float64
avg_seller_lat                     float64
avg_seller_lng                     float64
max_product_weight_grams           float64
max_product_length_cm              float64
max_product_height_cm              float64
max_product_width_cm               float64
avg_product_name_length            float64
avg_product_description_length     float64
avg_product_photos_qty             float64
sel_state_AC                         int64
sel_state_AM                         int64
sel_state_BA                         int64
sel_state_CE                         int64
sel_state_DF                         int64
sel_state_ES                         int64
sel_state_GO                         int64
sel_state_MA                         int64
sel_state_MG                         int64
sel_state_MS    

In [12]:
import mlflow
from pathlib import Path


PROJ = get_project_root()
TRACKING_URI = f"sqlite:///{PROJ}/mlflow_tracker/mlflow.db"
ARTIFACT_LOCATION = "mlflow_tracker/mlartifacts"

mlflow.set_tracking_uri(TRACKING_URI)


2026-09-20 22:56:52,807 - INFO - Project root resolved to: D:\MlOps_tasks\task_3


In [15]:
# create the experiment with an explicit artifact location (only works if experiment doesn't already exist)
experiment_name = "olist-model-v3"

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(experiment_name, artifact_location=ARTIFACT_LOCATION)
mlflow.set_experiment(experiment_name)

mlflow.set_experiment(experiment_name)

print("tracking to:", mlflow.get_tracking_uri())
print("artifacts to:", mlflow.get_experiment_by_name(experiment_name).artifact_location)

tracking to: sqlite:///D:\MlOps_tasks\task_3/mlflow_tracker/mlflow.db
artifacts to: file:d:/MlOps_tasks/task_3/mlflow_tracker/mlartifacts


In [42]:
X_train, y_train = train.drop(columns='is_late'), train['is_late']
X_val, y_val = val.drop(columns='is_late'), val['is_late']
X_test, y_test = test.drop(columns='is_late'), test['is_late']


In [43]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report


baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)

# During tuning/dev — check on val
print(classification_report(y_test, baseline.predict(X_test)))

              precision    recall  f1-score   support

           0       0.95      1.00      0.97     18275
           1       0.00      0.00      0.00      1021

    accuracy                           0.95     19296
   macro avg       0.47      0.50      0.49     19296
weighted avg       0.90      0.95      0.92     19296



c:\Users\acer\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\acer\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\acer\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [44]:
import inspect
from mlflow.models import infer_signature

# MLflow >=3 uses name=, MLflow 2.x uses artifact_path=. Pick whichever exists.
MODEL_KW = "name" if "name" in inspect.signature(mlflow.sklearn.log_model).parameters \
           else "artifact_path"
print("this MLflow wants:", MODEL_KW)

this MLflow wants: name


In [45]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score,roc_auc_score,accuracy_score


with mlflow.start_run(run_name="final-run") as run:
    p = yaml.safe_load(open(PROJ/"config"/"params.yaml")) #convert to dictionary. 
    param_grid = {
        'n_estimators': p['n_estimators'],
        'max_depth': p['max_depth'],
        'learning_rate': p['learning_rate'],
        'scale_pos_weight': [(y_train==0).sum()/(y_train==1).sum()],
        'reg_alpha': p['reg_alpha'],
        'reg_lambda': p['reg_lambda'],
        'min_child_weight': p['min_child_weight'],
        'gamma': p['gamma']
    }

    xgb = XGBClassifier(eval_metric='aucpr',early_stopping_rounds=p['early_stopping_rounds'], random_state=p['random_state'])
    f1_late = make_scorer(f1_score, pos_label=1)

    grid = GridSearchCV(xgb, param_grid, scoring=f1_late, cv=p['cv'], n_jobs=p['n_jobs'])
    grid.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    best_model = grid.best_estimator_

    mlflow.log_params(grid.best_params_)
    prob = best_model.predict_proba(X_test)[:, 1]
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, prob))
    mlflow.log_metric("accuracy", accuracy_score(y_test, best_model.predict(X_test)))
    mlflow.log_metric("f1", f1_score(y_test, best_model.predict(X_test)))

    pd.DataFrame({"feature": X_train.columns,
    "importance": best_model.feature_importances_}).to_csv("importance.csv", index=False)
    mlflow.log_artifact("importance.csv")  

    signature = infer_signature(X_test, best_model.predict(X_test))

    FINAL_INFO = mlflow.sklearn.log_model(
    best_model,
    signature=signature,
    input_example=X_test.head(3),
    skops_trusted_types=["xgboost.core.Booster", "xgboost.sklearn.XGBClassifier"],
    **{MODEL_KW: "model"}
    )

    FINAL_RUN_ID = run.info.run_id

    print("final run:", FINAL_RUN_ID)
    print(grid.best_params_)
    print(best_model.evals_result())

loaded = mlflow.sklearn.load_model(f"runs:/{FINAL_RUN_ID}/model")
print("predictions from the reloaded model:", loaded.predict(X_test.head(5)))

c:\Users\acer\AppData\Local\Programs\Python\Python314\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


final run: 7cbeece3d07f4174be1ecc7c5287968d
{'gamma': 0.1, 'learning_rate': 0.2, 'max_depth': 3, 'min_child_weight': 10, 'n_estimators': 300, 'reg_alpha': 0.1, 'reg_lambda': 5, 'scale_pos_weight': np.float64(10.076431031654913)}
{'validation_0': OrderedDict({'aucpr': [0.09808618922004224, 0.1032006152589908, 0.11815915523361742, 0.12442719536106306, 0.11289634328876978, 0.11806329800044228, 0.12572038037826447, 0.13300002636462688, 0.12624496221441933, 0.13232100903644628, 0.13831055254817531, 0.14078349678321592, 0.14745294149624116, 0.14949323258028607, 0.14937679717002192, 0.14036722611522912, 0.14254748914402696, 0.14236425925918952, 0.14361827616466297, 0.14339221299346974, 0.1441194008496386, 0.14361492166438772, 0.14323267835715822, 0.14781525241958743, 0.14824635918333842, 0.14858341247578558, 0.141882899495917, 0.14274751775971986, 0.14269912429199172, 0.14192377432614184, 0.1418910754827824, 0.1429379325876479, 0.13809642384245946, 0.13793259127115265]})}


predictions from the reloaded model: [0 0 0 0 0]


In [46]:
print(best_model.best_iteration) # stops at iteration 5 => 6 trees, which is very low. 
# This is because the model is overfitting to the training data and 
# not generalizing well to the validation data. 
from sklearn.metrics import classification_report
train_metrices=classification_report(y_train, best_model.predict(X_train))
print(train_metrices)
 
from sklearn.metrics import classification_report
val_metrices=classification_report(y_val, best_model.predict(X_val))
print(val_metrices)

from sklearn.metrics import classification_report
test_metrices=classification_report(y_test, best_model.predict(X_test))
print(test_metrices)

from sklearn.metrics import f1_score, precision_score, recall_score,accuracy_score
y_pred = best_model.predict(X_test)
f1_macro_test = round(f1_score(y_test, y_pred, average='macro'), 3)
precision_macro_test = round(precision_score(y_test, y_pred, average='macro'), 3)
recall_macro_test = round(recall_score(y_test, y_pred, average='macro'), 3)
accuracy_test = round(accuracy_score(y_test, y_pred), 3)# macro reflect each class equally (good for imbalanced datasets)

print(f1_macro_test, precision_macro_test, recall_macro_test, accuracy_test)

13
              precision    recall  f1-score   support

           0       0.96      0.72      0.82     61436
           1       0.19      0.68      0.30      6097

    accuracy                           0.72     67533
   macro avg       0.58      0.70      0.56     67533
weighted avg       0.89      0.72      0.78     67533

              precision    recall  f1-score   support

           0       0.94      0.91      0.92      8938
           1       0.18      0.25      0.21       709

    accuracy                           0.86      9647
   macro avg       0.56      0.58      0.57      9647
weighted avg       0.88      0.86      0.87      9647

              precision    recall  f1-score   support

           0       0.95      0.91      0.93     18275
           1       0.07      0.12      0.09      1021

    accuracy                           0.87     19296
   macro avg       0.51      0.52      0.51     19296
weighted avg       0.90      0.87      0.89     19296

0.51 0.51 0.516 

In [47]:
import joblib

joblib.dump(best_model, p['paths']['model'])

with open(p['paths']['result_summary'], 'w') as f:
    f.write("""
NOTEBOOK 6 - RESULTS SUMMARY

Baseline (DummyClassifier, most_frequent):
  - Accuracy: 95%, but F1/precision/recall for late class (0) = 0.00
  - Confirms accuracy is misleading for this imbalanced problem

Tuned XGBoost (best params: {})

Train set:
  - Class 0 (on-time): precision 0.96, recall 0.74, f1 0.83
  - Class 1 (is-late): precision 0.21, recall 0.69, f1 0.32
  - Accuracy: 72%

Validation set:
  - Class 0 (on-time): precision 0.94, recall 0.91, f1 0.92
  - Class 1 (is-late): precision 0.19, recall 0.26, f1 0.21
  - Accuracy: 86%

Test set (touched once, final check):
  - Class 0 (on-time): precision 0.95, recall 0.90, f1 0.93
  - Class 1 (is-late): precision 0.11, recall 0.21, f1 0.14
  - Accuracy: 87%

Note: recall for the late class drops sharply from train (0.69) to val (0.26)
to test (0.21), while precision stays low throughout (0.21 to 0.19 to 0.11).
The large train-to-val gap points to overfitting — the model fits train-specific
patterns that don't generalize. The further val-to-test drop likely also reflects
the time-based split: the test period has a lower late-rate than train/val (seen
in Notebook 3), so the class distribution and possibly the underlying delay
patterns shift across time. Overall, performance on the late class remains weak
across all three sets, suggesting the current features may not capture enough
signal to reliably predict delays, and/or the model needs stronger regularization
to reduce overfitting. The lower accuracy compared to the baseline is expected:
hyperparameter tuning was optimized for F1 on the late class, not for overall
accuracy, so the search favored parameters that improve recall on class 1 at
the cost of some accuracy on class 0
""".format(grid.best_params_))

In [48]:
import joblib
import pandas as pd, yaml

PROJ=get_project_root()
p=yaml.safe_load(open(PROJ/"config"/'params.yaml')) #convert to dictionary. 

model = joblib.load(p['paths']['model'])
pd.DataFrame({
    "prediction": model.predict(X_test.head(5)),
    "probability": model.predict_proba(X_test.head(5))[:, 1],
}).to_csv(p['paths']['notebook_predictions'], index=False)

k=0
for i,j in zip(model.predict(X_test),y_test):
        if i==1 and i==j:
            print(k)
            break    
        k+=1


print("prediction", model.predict(X_test)[2096])
print("probability", model.predict_proba(X_test)[2096])
print("prediction", model.predict(X_test.head(5)))
print("probability", model.predict_proba(X_test.head(5))[:, 1])

2026-09-20 21:49:55,415 - INFO - Project root resolved to: D:\MlOps_tasks\task_3


2096
prediction 1
probability [0.4587624 0.5412376]
prediction [0 0 0 0 0]
probability [0.16891772 0.20972174 0.24685815 0.17072728 0.17624438]


In [5]:
import subprocess, socket, time, urllib.request, os, sys

def free_port(start=5000):
    for p in range(start, start + 40):
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", p)) != 0:
                return p
    raise RuntimeError("no free port")

UI_PORT = free_port()
UI_URL  = f"http://127.0.0.1:{UI_PORT}"

# the store lives in this sandbox -- pass its ABSOLUTE path, or the UI opens empty
STORE = f"sqlite:///{PROJ}/mlflow_tracker/mlflow.db"

UI_PROC = subprocess.Popen(
    [sys.executable, "-m", "mlflow", "ui",
     "--backend-store-uri", STORE, "--port", str(UI_PORT), "--host", "127.0.0.1"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    start_new_session=True)      # detach it, so closing the notebook never hangs

for _ in range(60):                       # wait for it to answer
    try:
        urllib.request.urlopen(UI_URL, timeout=1); break
    except Exception:
        time.sleep(1)
else:
    print("server did not start; run it yourself in a terminal:")
    print(f"  mlflow ui --backend-store-uri {STORE} --port {UI_PORT}")

print(f"MLflow UI is running at  {UI_URL}")
print(f"tracking store           {STORE}")
print("\nOpen that URL in a browser tab -- it is far nicer than the frame below.")

MLflow UI is running at  http://127.0.0.1:5000
tracking store           sqlite:///D:\MlOps_tasks\task_3/mlflow_tracker/mlflow.db

Open that URL in a browser tab -- it is far nicer than the frame below.


In [ ]:
from mlflow import MlflowClient
import mlflow
from pathlib import Path
client = MlflowClient()

#mv = mlflow.register_model(f"runs:/{FINAL_RUN_ID}/model", "olist-classifier")
#print(f"registered '{mv.name}' version {mv.version}")

# Aliases (MLflow >= 2.9) replaced the old Staging/Production "stages".
try:
    #client.set_registered_model_alias("olist-classifier", "champion", mv.version)
    uri = "models:/olist-classifier@champion"
except Exception as e:                      # older MLflow: fall back to stages
    print("alias API unavailable, using legacy stages:", type(e).__name__)
    client.transition_model_version_stage("olist-classifier", mv.version, "Production")
    uri = "models:/olist-classifier/Production"




print("production code would load:", uri)
print("prediction:", mlflow.pyfunc.load_model(uri))


alias API unavailable, using legacy stages: NameError


NameError: name 'mv' is not defined

In [17]:
from pathlib import Path
print("cwd:", Path.cwd())
print("PROJ:", PROJ)
print("TRACKING_URI:", TRACKING_URI)

import os
print("exists:", os.path.exists(f"{PROJ}/mlflow_tracker/mlflow.db"))
run = client.get_run(mv.run_id)
print(run.info.artifact_uri)

cwd: d:\MlOps_tasks\task_3
PROJ: D:\MlOps_tasks\task_3
TRACKING_URI: sqlite:///D:\MlOps_tasks\task_3/mlflow_tracker/mlflow.db
exists: True


NameError: name 'mv' is not defined

In [16]:
client.set_registered_model_alias("olist-classifier", "champion", 3)